# Phase B: BirdNET Baseline
Run a pre-trained bird recognition model on our data -- zero training needed.

In [2]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from birdnetlib import Recording
from birdnetlib.analyzer import Analyzer
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = '/data/scratch/scienceteam/jupyter-mir/bird/Data/birdclef-2026'
TRAIN_AUDIO = os.path.join(BASE_DIR, 'train_audio')
TRAIN_SOUNDSCAPES = os.path.join(BASE_DIR, 'train_soundscapes')

analyzer = Analyzer()
print("BirdNET loaded successfully.")

Labels loaded.
load model True
Model loaded.
Labels loaded.
load_species_list_model
Meta model loaded.
BirdNET loaded successfully.


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


## 1. Test on a single clean clip

In [3]:
train = pd.read_csv(os.path.join(BASE_DIR, 'train.csv'))
taxonomy = pd.read_csv(os.path.join(BASE_DIR, 'taxonomy.csv'))

sample = train[train.class_name == 'Aves'].sample(1, random_state=42).iloc[0]
filepath = os.path.join(TRAIN_AUDIO, sample.filename)

recording = Recording(analyzer, filepath,
                      lat=-19.0, lon=-56.5,  # Pantanal center
                      date=datetime(month=6, day=1, year=2025),
                      min_conf=0.1)
recording.analyze()

print(f"File: {sample.filename}")
print(f"True label: {sample.primary_label} ({sample.common_name})")
print(f"\nBirdNET detections ({len(recording.detections)}):")
for d in recording.detections:
    print(f"  {d['start_time']:.0f}-{d['end_time']:.0f}s: {d['common_name']} ({d['scientific_name']}) conf={d['confidence']:.3f}")

read_audio_data
read_audio_data: complete, read  9 chunks.
analyze_recording iNat768993.ogg
recording has lon/lat
set_predicted_species_list_from_position
return_predicted_species_list
20
1067 species loaded.
File: fepowl/iNat768993.ogg
True label: fepowl (Ferruginous Pygmy Owl)

BirdNET detections (20):
  0-3s: Monk Parakeet (Myiopsitta monachus) conf=0.163
  3-6s: Ferruginous Pygmy-Owl (Glaucidium brasilianum) conf=0.466
  3-6s: White-throated Kingbird (Tyrannus albogularis) conf=0.195
  6-9s: Monk Parakeet (Myiopsitta monachus) conf=0.584
  6-9s: Ferruginous Pygmy-Owl (Glaucidium brasilianum) conf=0.533
  6-9s: White-throated Kingbird (Tyrannus albogularis) conf=0.325
  9-12s: Monk Parakeet (Myiopsitta monachus) conf=0.849
  12-15s: Monk Parakeet (Myiopsitta monachus) conf=0.524
  12-15s: Yellow-chevroned Parakeet (Brotogeris chiriri) conf=0.295
  15-18s: Ferruginous Pygmy-Owl (Glaucidium brasilianum) conf=0.451
  15-18s: White-wedged Piculet (Picumnus albosquamatus) conf=0.202
  15

## 2. Run BirdNET on a sample of training clips
How often does it get the right species?

In [4]:
birds = train[train.class_name == 'Aves'].sample(200, random_state=42)
correct = 0
detected = 0
results = []

for _, row in tqdm(birds.iterrows(), total=len(birds), desc="Running BirdNET"):
    filepath = os.path.join(TRAIN_AUDIO, row.filename)
    if not os.path.exists(filepath):
        continue
    try:
        rec = Recording(analyzer, filepath,
                        lat=-19.0, lon=-56.5,
                        date=datetime(month=6, day=1, year=2025),
                        min_conf=0.1)
        rec.analyze()
        
        predicted_species = set()
        top_conf = 0
        for d in rec.detections:
            predicted_species.add(d['scientific_name'].lower())
            top_conf = max(top_conf, d['confidence'])
        
        true_sci = row.scientific_name.lower()
        hit = true_sci in predicted_species
        if hit:
            correct += 1
        if len(rec.detections) > 0:
            detected += 1
            
        results.append({
            'primary_label': row.primary_label,
            'true_species': row.scientific_name,
            'n_detections': len(rec.detections),
            'correct': hit,
            'top_conf': top_conf
        })
    except Exception as e:
        continue

results_df = pd.DataFrame(results)
print(f"\nResults on {len(results_df)} bird clips:")
print(f"  Clips with any detection: {detected} ({detected/len(results_df)*100:.1f}%)")
print(f"  Correct species detected: {correct} ({correct/len(results_df)*100:.1f}%)")

Running BirdNET:   0%|          | 0/200 [00:00<?, ?it/s]

read_audio_data
read_audio_data: complete, read  9 chunks.
analyze_recording iNat768993.ogg
recording has lon/lat
set_predicted_species_list_from_position


Running BirdNET:   1%|          | 2/200 [00:00<01:11,  2.75it/s]

read_audio_data
read_audio_data: complete, read  3 chunks.
analyze_recording XC728465.ogg
recording has lon/lat
set_predicted_species_list_from_position
read_audio_data


Running BirdNET:   2%|▏         | 3/200 [00:00<00:50,  3.88it/s]

...

read_audio_data
read_audio_data: complete, read  15 chunks.
analyze_recording XC564692.ogg
recording has lon/lat
set_predicted_species_list_from_position


Running BirdNET: 100%|█████████▉| 199/200 [02:33<00:00,  1.50it/s]

read_audio_data
read_audio_data: complete, read  13 chunks.
analyze_recording XC465140.ogg
recording has lon/lat
set_predicted_species_list_from_position


Running BirdNET: 100%|██████████| 200/200 [02:33<00:00,  1.30it/s]


Results on 200 bird clips:
  Clips with any detection: 182 (91.0%)
  Correct species detected: 166 (83.0%)


## 3. Run BirdNET on labeled soundscapes
This is closer to the actual competition task.

In [5]:
ts_labels = pd.read_csv(os.path.join(BASE_DIR, 'train_soundscapes_labels.csv'))
ts_files = ts_labels.filename.unique()[:10]  # first 10 soundscapes

species_to_label = dict(zip(taxonomy.scientific_name.str.lower(), taxonomy.primary_label))

all_results = []

for ts_file in tqdm(ts_files, desc="Soundscapes"):
    filepath = os.path.join(TRAIN_SOUNDSCAPES, ts_file)
    if not os.path.exists(filepath):
        continue
    try:
        rec = Recording(analyzer, filepath,
                        lat=-19.0, lon=-56.5,
                        date=datetime(month=6, day=1, year=2025),
                        min_conf=0.1)
        rec.analyze()
        
        # Get true labels for this file
        file_labels = ts_labels[ts_labels.filename == ts_file]
        true_species = set()
        for labels in file_labels.primary_label:
            true_species.update(labels.split(';'))
        
        # Get BirdNET predictions
        predicted_labels = set()
        for d in rec.detections:
            sci = d['scientific_name'].lower()
            if sci in species_to_label:
                predicted_labels.add(species_to_label[sci])
        
        true_birds = true_species & set(taxonomy[taxonomy.class_name == 'Aves'].primary_label)
        hits = predicted_labels & true_species
        
        all_results.append({
            'file': ts_file,
            'true_species': len(true_species),
            'true_birds': len(true_birds),
            'birdnet_detections': len(rec.detections),
            'birdnet_unique': len(predicted_labels),
            'correct': len(hits),
            'missed': len(true_species - predicted_labels),
            'false_pos': len(predicted_labels - true_species)
        })
        
    except Exception as e:
        print(f"Error on {ts_file}: {e}")
        continue

results_df = pd.DataFrame(all_results)
print("\n=== BirdNET on Soundscapes ===")
print(results_df.to_string(index=False))
print(f"\nAverage: {results_df.correct.mean():.1f} correct, {results_df.missed.mean():.1f} missed, {results_df.false_pos.mean():.1f} false positives per file")

Soundscapes:   0%|          | 0/10 [00:00<?, ?it/s]

read_audio_data
read_audio_data: complete, read  20 chunks.
analyze_recording BC2026_Train_0039_S22_20211231_201500.ogg
recording has lon/lat
set_predicted_species_list_from_position


Soundscapes:  10%|█         | 1/10 [00:01<00:11,  1.27s/it]

read_audio_data
read_audio_data: complete, read  20 chunks.
analyze_recording BC2026_Train_0019_S22_20211104_200000.ogg
recording has lon/lat
set_predicted_species_list_from_position


Soundscapes:  20%|██        | 2/10 [00:02<00:10,  1.26s/it]

...

read_audio_data
read_audio_data: complete, read  20 chunks.
analyze_recording BC2026_Train_0046_S22_20220127_230000.ogg
recording has lon/lat
set_predicted_species_list_from_position


Soundscapes:  90%|█████████ | 9/10 [00:11<00:01,  1.28s/it]

read_audio_data
read_audio_data: complete, read  20 chunks.
analyze_recording BC2026_Train_0057_S15_20250617_060000.ogg
recording has lon/lat
set_predicted_species_list_from_position


Soundscapes: 100%|██████████| 10/10 [00:12<00:00,  1.27s/it]


=== BirdNET on Soundscapes ===
                                     file  true_species  true_birds  birdnet_detections  birdnet_unique  correct  missed  false_pos
BC2026_Train_0039_S22_20211231_201500.ogg             5           0                   0               0        0       5          0
BC2026_Train_0019_S22_20211104_200000.ogg             5           1                   7               1        0       5          1
BC2026_Train_0053_S22_20220211_204500.ogg             6           0                   7               1        0       6          1
BC2026_Train_0047_S22_20220131_003000.ogg             6           0                   1               0        0       6          0
BC2026_Train_0027_S22_20211129_014500.ogg             8           1                   9               2        0       8          2
BC2026_Train_0005_S08_20250607_070007.ogg             6           1                  15               1        0       6          1
BC2026_Train_0014_S13_20220228_214500.ogg   

## 4. BirdNET Limitations
What this baseline CAN'T do (and why we need a custom model).

In [6]:
# How many of our 234 target species are birds?
bird_count = taxonomy[taxonomy.class_name == 'Aves'].shape[0]
non_bird_count = taxonomy[taxonomy.class_name != 'Aves'].shape[0]

print("=== Why BirdNET is not enough ===")
print(f"  Target species:    234")
print(f"  Birds (Aves):      {bird_count} -- BirdNET can attempt these")
print(f"  Non-birds:         {non_bird_count} -- BirdNET CANNOT detect these:")
for cls in taxonomy[taxonomy.class_name != 'Aves'].class_name.unique():
    count = taxonomy[taxonomy.class_name == cls].shape[0]
    print(f"    {cls}: {count} species")
print(f"\n  BirdNET covers at most {bird_count}/234 = {bird_count/234*100:.0f}% of target species")
print(f"  Missing {non_bird_count} species = guaranteed zero score on those")
print(f"\n  --> We need a custom model trained on ALL 234 species.")

=== Why BirdNET is not enough ===
  Target species:    234
  Birds (Aves):      162 -- BirdNET can attempt these
  Non-birds:         72 -- BirdNET CANNOT detect these:
    Insecta: 28 species
    Reptilia: 1 species
    Amphibia: 35 species
    Mammalia: 8 species

  BirdNET covers at most 162/234 = 69% of target species
  Missing 72 species = guaranteed zero score on those

  --> We need a custom model trained on ALL 234 species.
